# Project 8 – Dashboard (OSS4008A)
**Notebook 02:** the web dashboard for Project 8, laid out from the course's *Draft for the project UI schema*:
AI assistant / analysis panel on the left, interactive map on the right, project buttons along the bottom.

**How to run:** Runtime → Run all, then click the link printed by the last cell.
The link works while this Colab session is open; every new session gives a new link.

**Data:** OurAirports (public domain), cleaned in notebook 01 and read from the course GitHub repository;
Ontario Imagery Web Map Service, Land Information Ontario (Open Government Licence – Ontario).

In [ ]:
# Cell 1 – Install Streamlit (web app framework), folium (maps) and streamlit-folium (map ↔ app link)
!pip install -q streamlit streamlit-folium folium

## The app
The next cell **saves** the whole app as `app.py` (`%%writefile` writes the cell to a file instead of running it).
To change the app, edit this cell, then run Cells 2, 3 and 4 again. The link from Cell 5 stays the same.

In [ ]:
%%writefile app.py
# app.py – OSS4008A shared dashboard, Project 8: Airports and airplanes in Canada
# Layout follows the course's "Draft for the project UI schema":
#   left  = AI assistant / data analysis panel
#   right = interactive map (zoom, click a point for info, draw a box for an area summary)
#   bottom = one button per class project (only Project 8 is active in this app)
# Runs on Google Colab with Streamlit. All data is open:
#   - OurAirports (public domain), cleaned in notebook 01 and stored in the course GitHub repo
#   - Ontario Imagery Web Map Service, Land Information Ontario (Open Government Licence – Ontario)

import copy                                  # copy.deepcopy gives each page run a fresh copy of the saved map
import pandas as pd
import streamlit as st
import folium
from folium.plugins import Draw              # adds the "draw a rectangle" tool to the map
from streamlit_folium import st_folium       # shows a folium map in Streamlit and returns what the user drew

st.set_page_config(page_title="OSS4008A Remote Sensing Dashboard", layout="wide")

# ---------------------------------------------------------------- data and settings
# Cleaned airport table from notebook 01 (read straight from the course GitHub repository)
CSV_URL = ("https://raw.githubusercontent.com/AdityaR28/"
           "OSS4008A---Introduction-to-Remote-Sensing-and-Applications/main/"
           "students/nifemi_ogunmeru/data/canada_airports_clean.csv")

# Ontario aerial imagery (~20 cm per pixel), served as ready-made map tiles. ArcGIS orders tiles as z/y/x.
ONTARIO_TILES = ("https://ws.lioservices.lrc.gov.on.ca/arcgis2/rest/services/"
                 "LIO_Imagery/Ontario_Imagery_Web_Map_Service/MapServer/tile/{z}/{y}/{x}")
ONTARIO_ATTR = "Contains information licensed under the Open Government Licence – Ontario"

# Colour, dot size and label for each airport type (same colours as notebook 01)
STYLES = {
    "large_airport":  {"color": "#d7191c", "radius": 7, "label": "Large airports"},
    "medium_airport": {"color": "#fdae61", "radius": 5, "label": "Medium airports"},
    "small_airport":  {"color": "#2b83ba", "radius": 3, "label": "Small airports"},
    "seaplane_base":  {"color": "#5ab4ac", "radius": 3, "label": "Seaplane bases"},
    "heliport":       {"color": "#7b3294", "radius": 3, "label": "Heliports"},
}

PROVINCES = {"AB": "Alberta", "BC": "British Columbia", "MB": "Manitoba", "NB": "New Brunswick",
             "NL": "Newfoundland and Labrador", "NS": "Nova Scotia", "NT": "Northwest Territories",
             "NU": "Nunavut", "ON": "Ontario", "PE": "Prince Edward Island", "QC": "Quebec",
             "SK": "Saskatchewan", "YT": "Yukon"}

# Bottom buttons: the class projects from the course document (number, short label)
PROJECTS = [(1, "Homes / WUI"), (2, "Factories"), (3, "Oil & pipes"), (4, "Railways"),
            (5, "Hospitals"), (8, "Airports ✈"), (9, "Ships & ports"), (11, "Water"),
            (12, "Real estate"), (13, "Farms"), (22, "Disasters"), (23, "Wildfires"), (24, "Floods")]
MY_PROJECT = 8

CANADA_VIEW = {"center": [56.0, -96.0], "zoom": 4}


@st.cache_data                     # download the CSV once, not on every click
def load_airports():
    return pd.read_csv(CSV_URL)


airports = load_airports()

# st.session_state keeps values between reruns (Streamlit re-runs this whole file after every click)
if "view" not in st.session_state:
    st.session_state.view = dict(CANADA_VIEW)
if "chat" not in st.session_state:
    st.session_state.chat = [{"role": "assistant",
                              "content": "Hi! Ask me about Canada's airports, or type **help**."}]


# ---------------------------------------------------------------- helper functions
def text(value):
    """Show missing values (NaN) as a dash instead of 'nan'."""
    return "—" if pd.isna(value) or value == "" else str(value)


def type_geojson(t):
    """All airports of one type as one compact GeoJSON layer (much smaller and faster than 2,477 separate markers).
    The properties hold the text shown in the popup ('click on pixel → information' in the draft)."""
    features = []
    for _, row in airports[airports["type"] == t].iterrows():
        elev = "—" if pd.isna(row["elevation_ft"]) else f"{int(row['elevation_ft'])} ft"
        features.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [float(row["lon"]), float(row["lat"])]},   # GeoJSON = [lon, lat]
            "properties": {"name": row["name"],
                           "kind": STYLES[t]["label"][:-1],                     # "Large airports" -> "Large airport"
                           "codes": f"{text(row['iata_code'])} / {text(row['icao_code'])}",
                           "place": f"{text(row['municipality'])}, {row['province']}",
                           "elev": elev,
                           "sched": text(row["scheduled_service"])},
        })
    return {"type": "FeatureCollection", "features": features}


@st.cache_resource      # build the map ONCE (a clean master copy); moving the view is done by st_folium(center=..., zoom=...)
def build_map():
    """Basemap, Ontario aerial layer, one layer per airport type, draw tool."""
    m = folium.Map(location=CANADA_VIEW["center"], zoom_start=CANADA_VIEW["zoom"], max_zoom=20,
                   control_scale=True, prefer_canvas=True)   # canvas draws thousands of dots much faster

    # Aerial imagery first, so the airport dots sit on top.
    # min_zoom=12: the layer stays switched on but only loads when you zoom in to airport level.
    folium.TileLayer(tiles=ONTARIO_TILES, attr=ONTARIO_ATTR, name="Ontario aerial imagery (~20 cm)",
                     overlay=True, show=True, min_zoom=12, max_zoom=20).add_to(m)

    # Airport layers, drawn small → large so large airports end up on top
    for t in reversed(list(STYLES)):
        s = STYLES[t]
        folium.GeoJson(
            type_geojson(t),
            name=s["label"],
            show=(t not in ["heliport", "seaplane_base"]),
            marker=folium.CircleMarker(radius=s["radius"], fill=True),
            style_function=lambda f, c=s["color"]: {"color": c, "fillColor": c,
                                                     "fillOpacity": 0.85, "weight": 1},
            tooltip=folium.GeoJsonTooltip(fields=["name"], labels=False),
            popup=folium.GeoJsonPopup(fields=["name", "kind", "codes", "place", "elev", "sched"],
                                      aliases=["Name", "Type", "IATA / ICAO", "Location",
                                               "Elevation", "Scheduled service"]),
        ).add_to(m)

    # Rectangle-only draw tool ("drawing a box" in the draft)
    Draw(draw_options={"rectangle": True, "polyline": False, "polygon": False, "circle": False,
                       "marker": False, "circlemarker": False},
         edit_options={"edit": False}).add_to(m)
    folium.LayerControl(collapsed=True).add_to(m)
    return m


def airports_in_box(drawing):
    """Return the airports inside a rectangle drawn on the map (GeoJSON coordinates are [lon, lat])."""
    coords = drawing["geometry"]["coordinates"][0]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    inside = (airports["lon"].between(min(lons), max(lons)) &
              airports["lat"].between(min(lats), max(lats)))
    return airports[inside]


def answer(question):
    """Simple rule-based assistant (a placeholder for the AI model planned for Progress 3)."""
    q = question.strip()
    ql = q.lower()

    if ql in ["help", "?"]:
        return ("Try:\n- **YOW** (or any airport code) – zoom to that airport\n"
                "- **how many airports in ON** (province code in capitals, or the full name)\n"
                "- **how many heliports** / large / medium / small / seaplane\n"
                "- **reset** – back to all of Canada\n"
                "- Draw a box on the map (square tool) for an area summary")

    if ql == "reset":
        st.session_state.view = dict(CANADA_VIEW)
        return "Back to the whole of Canada."

    if "how many" in ql:
        # Province (code or full name)
        for code, name in PROVINCES.items():
            if code in q.replace("?", " ").split() or name.lower() in ql:   # codes must be capitals (ON, not "on")
                n = int((airports["province"] == code).sum())
                return f"**{n}** active facilities in {name} (airports, heliports and seaplane bases)."
        # Facility type
        for key, t in [("heliport", "heliport"), ("seaplane", "seaplane_base"), ("large", "large_airport"),
                       ("medium", "medium_airport"), ("small", "small_airport")]:
            if key in ql:
                n = int((airports["type"] == t).sum())
                return f"**{n}** {STYLES[t]['label'].lower()} in Canada."
        return f"**{len(airports):,}** active facilities in Canada."

    # Airport codes: 3 letters (IATA, e.g. YOW) or 4 (ICAO, e.g. CYOW)
    for word in q.upper().replace("?", " ").replace(",", " ").split():
        match = airports[(airports["iata_code"] == word) | (airports["icao_code"] == word) |
                         (airports["ident"] == word)]
        if len(match) > 0:
            a = match.iloc[0]
            st.session_state.view = {"center": [float(a["lat"]), float(a["lon"])], "zoom": 16}
            extra = (" The aerial layer is on, so zoom in to see parked planes."
                     if a["province"] == "ON" else
                     " Sharp aerial imagery is only available in Ontario for now.")
            return f"Zoomed to **{a['name']}** ({text(a['iata_code'])}), {text(a['municipality'])}, {a['province']}.{extra}"

    return "I can't answer that yet. Type **help** to see what I understand. (The AI model arrives in Progress 3.)"


# ---------------------------------------------------------------- page layout
# Streamlit greys out the page while it works ("stale" elements). This style rule keeps everything fully visible.
st.markdown("<style>[data-stale='true'], .stale-element {opacity: 1 !important; filter: none !important;}</style>",
            unsafe_allow_html=True)

st.markdown("### 🛰️ OSS4008A Remote Sensing Dashboard  ·  Project 8: Airports and airplanes in Canada")
left, right = st.columns([1, 2.2], gap="medium")
st.divider()
bottom = st.container()     # created now so it appears at the bottom; filled first (see step 1)

# Order matters for speed: buttons and chat are handled BEFORE the map is drawn,
# so the map already shows the new view and the page only runs once per action.

# 1) BOTTOM: one button per class project (others will be connected when the projects merge)
with bottom:
    cols = st.columns(len(PROJECTS))
    for col, (num, label) in zip(cols, PROJECTS):
        mine = (num == MY_PROJECT)
        if col.button(label, key=f"project_{num}", disabled=not mine,
                      type="primary" if mine else "secondary",
                      help=f"Project {num}" + ("" if mine else " (connected when projects merge)")):
            st.session_state.view = dict(CANADA_VIEW)      # pressing a project button reloads its map view

# 2) LEFT: assistant panel. Areas are reserved now and filled after the map (step 4).
with left:
    st.markdown("#### 🤖 AI assistant")
    summary_area = st.container()
    chat_area = st.container(height=230)
    prompt = st.chat_input("Ask about airports… (try: YOW)")
    if prompt:
        st.session_state.chat.append({"role": "user", "content": prompt})
        st.session_state.chat.append({"role": "assistant", "content": answer(prompt)})

# 3) RIGHT: the interactive map. The map itself is built once (cached); only centre and zoom change.
with right:
    view = st.session_state.view
    # returned_objects keeps the app fast: only the drawn box is sent back, not every pan or zoom
    # Draw a fresh COPY of the master map: drawing a map changes it slightly, so reusing the same object
    # after a page refresh would produce broken map code (the map collapsed to a thin strip).
    map_data = st_folium(copy.deepcopy(build_map()), center=view["center"], zoom=view["zoom"],
                         height=620, use_container_width=True,
                         returned_objects=["last_active_drawing"], key="map")

# 4) LEFT (continued): area summary and chat history
with summary_area:
    drawing = (map_data or {}).get("last_active_drawing")
    if drawing and drawing.get("geometry", {}).get("type") == "Polygon":
        # Summary of the box the user drew
        sel = airports_in_box(drawing)
        st.markdown("**Selected area**")
        c1, c2 = st.columns(2)
        c1.metric("Facilities", len(sel))
        c2.metric("Planes detected", "—", help="The plane-detection model is added in Progress 3")
        if len(sel) > 0:
            counts = sel["type"].map(lambda t: STYLES[t]["label"]).value_counts()
            st.bar_chart(counts, horizontal=True, height=180)
            main = sel[sel["type"].isin(["large_airport", "medium_airport"])]
            if len(main) > 0:
                st.dataframe(main[["name", "iata_code", "municipality", "province"]],
                             hide_index=True, height=160)
    else:
        # Canada-wide overview until a box is drawn
        st.markdown("**All of Canada**: draw a box on the map for an area summary")
        c1, c2 = st.columns(2)
        c1.metric("Facilities", f"{len(airports):,}")
        c2.metric("Large airports", int((airports["type"] == "large_airport").sum()))
        counts = airports["type"].map(lambda t: STYLES[t]["label"]).value_counts()
        st.bar_chart(counts, horizontal=True, height=180)

with chat_area:
    for msg in st.session_state.chat:
        st.chat_message(msg["role"]).markdown(msg["content"])

In [ ]:
# Cell 3 – Check that app.py is the dashboard (not an old test file) and has no Python syntax errors
src = open('app.py', encoding='utf-8').read()
assert 'OSS4008A shared dashboard' in src, 'app.py is not the dashboard – rerun Cell 2'
compile(src, 'app.py', 'exec')            # raises an error here if the code has a syntax mistake
print(f'app.py OK ✅  ({len(src.splitlines())} lines)')

In [ ]:
# Cell 4 – (Re)start the Streamlit server in the background and wait until it answers
import time, urllib.request

!pkill -f "streamlit run" || true          # stop any copy that is already running (avoids "port not available")
time.sleep(2)
!nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit_log.txt 2>&1 &

ok = False
for _ in range(30):                        # check once per second, for up to 30 seconds
    time.sleep(1)
    try:
        ok = urllib.request.urlopen('http://localhost:8501/_stcore/health').read().decode() == 'ok'
        if ok:
            break
    except Exception:
        pass                               # server not ready yet

if ok:
    print('Streamlit server is running ✅')
else:
    print('❌ Streamlit did not start. Server log:')
    print(open('streamlit_log.txt').read())

In [ ]:
# Cell 5 – Public link to the app through a free Cloudflare quick tunnel (no account needed)
import os, re

if not os.path.exists('cloudflared'):      # download the tunnel program once per session
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

!pkill -f "cloudflared tunnel" || true     # close any old tunnel
!nohup ./cloudflared tunnel --url http://localhost:8501 --no-autoupdate > tunnel_log.txt 2>&1 &

url = None
for _ in range(30):                        # wait up to 30 seconds for the link to appear in the log
    time.sleep(1)
    found = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel_log.txt').read())
    if found:
        url = found[0]
        break

print('Open the dashboard here 👉', url if url else 'no link yet – run this cell again')
print('If the page shows an error at first, wait 15–20 seconds and refresh.')